### Imports

In [3]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


### Data cleaning

In [4]:
X_train = pd.read_csv('../data/X_train.csv', index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv', index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv', index_col='ROW_ID')
sample_submission = pd.read_csv('../submissions/sample_submission.csv', index_col='ROW_ID')

# La cible est binaire
y_train_clf = (y_train['target'] > 0).astype(int)

# Gestion des valeurs manquantes
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

### Transformation Tabulaire $\rightarrow$ Séquentielle (3D)

In [5]:
# On prend l'ordre inverse pour aller du plus ancien (J-20) au plus récent (J-1)
ret_cols = [f'RET_{i}' for i in range(20, 0, -1)]
vol_cols = [f'SIGNED_VOLUME_{i}' for i in range(20, 0, -1)]

# Extraction et Standardisation optionnelle (souvent utile pour les Réseaux de Neurones)
# Pour faire simple, on passe les données brutes car elles sont déjà dans de petites échelles
X_train_ret = X_train[ret_cols].values
X_train_vol = X_train[vol_cols].values

X_test_ret = X_test[ret_cols].values
X_test_vol = X_test[vol_cols].values

# Empilement pour obtenir la dimension : (N_samples, 20_jours, 2_features)
X_train_seq = np.stack([X_train_ret, X_train_vol], axis=-1)
X_test_seq = np.stack([X_test_ret, X_test_vol], axis=-1)

print(f"Shape de X_train_seq : {X_train_seq.shape}") 
# Devrait afficher : (527073, 20, 2)

Shape de X_train_seq : (527073, 20, 2)


### Architecture du Modèle (CNN 1D + LSTM)

In [7]:
def build_model():
    model = Sequential([
        # Couche de Convolution 1D pour extraire des motifs locaux sur 3 jours glissants
        Conv1D(filters=16, kernel_size=3, activation='relu', input_shape=(20, 2)),
        BatchNormalization(),
        Dropout(0.3),
        
        # Couche LSTM pour lire la séquence temporelle
        LSTM(16, return_sequences=False),
        Dropout(0.3),
        
        # Couche Dense finale
        Dense(8, activation='relu'),
        Dense(1, activation='sigmoid') # Sigmoid car classification binaire
    ])
    
    model.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

# Afficher un résumé de l'architecture
temp_model = build_model()
temp_model.summary()

/home/carl/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-27 14:37:19.291174: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 18, 16)         │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 18, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 18, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 16)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,433 (9.50 KB)

 Trainable params: 2,401 (9.38 KB)

 Non-trainable params: 32 (128.00 B)

### Entraînement avec GroupKFold sur le Timestamp (TS)

In [8]:
train_dates = X_train['TS']
unique_dates = train_dates.unique()

n_splits = 5
gkf = GroupKFold(n_splits=n_splits)

scores_nn = []
models_nn = []

print("Début de l'entraînement...")

# GroupKFold se base sur X_train, y_train_clf, et les groupes (train_dates)
for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train_seq, y_train_clf, groups=train_dates)):
    print(f"\n--- Fold {fold + 1} ---")
    
    X_tr, y_tr = X_train_seq[train_idx], y_train_clf.iloc[train_idx]
    X_val, y_val = X_train_seq[val_idx], y_train_clf.iloc[val_idx]
    
    model = build_model()
    
    # EarlyStopping arrête l'entraînement si l'accuracy de validation ne s'améliore plus
    es = EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True)
    
    model.fit(X_tr, y_tr, 
              validation_data=(X_val, y_val),
              epochs=15, 
              batch_size=1024, # Grand batch size = plus stable et rapide
              callbacks=[es],
              verbose=1)
    
    # Prédiction sur le set de validation
    y_val_pred_proba = model.predict(X_val, batch_size=2048).ravel()
    y_val_pred = (y_val_pred_proba > 0.5).astype(int)
    
    acc = accuracy_score(y_val, y_val_pred)
    scores_nn.append(acc)
    models_nn.append(model)
    
    print(f"Accuracy Fold {fold + 1}: {acc * 100:.2f}%")

mean_acc = np.mean(scores_nn) * 100
print(f"\nAccuracy Moyenne de la Validation Croisée : {mean_acc:.2f}%")

Début de l'entraînement...

--- Fold 1 ---


/home/carl/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-27 14:37:32.684505: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67465760 exceeds 10% of free system memory.


Epoch 1/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 15s 27ms/step - accuracy: 0.5042 - loss: 0.6933 - val_accuracy: 0.5084 - val_loss: 0.6930
Epoch 2/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.5062 - loss: 0.6931 - val_accuracy: 0.5085 - val_loss: 0.6930
Epoch 3/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 15s 35ms/step - accuracy: 0.5070 - loss: 0.6930 - val_accuracy: 0.5085 - val_loss: 0.6930
Epoch 4/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.5073 - loss: 0.6930 - val_accuracy: 0.5073 - val_loss: 0.6930
Epoch 5/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5070 - loss: 0.6930 - val_accuracy: 0.5058 - val_loss: 0.6930
Epoch 6/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5074 - loss: 0.6930 - val_accuracy: 0.5070 - val_loss: 0.6930
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
Accuracy Fold 1: 50.85%

--- Fold 2 ---


/home/carl/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15


...

412/412 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - accuracy: 0.5041 - loss: 0.6939 - val_accuracy: 0.5097 - val_loss: 0.6929
Epoch 2/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.5062 - loss: 0.6931 - val_accuracy: 0.5103 - val_loss: 0.6929
Epoch 3/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5060 - loss: 0.6931 - val_accuracy: 0.5108 - val_loss: 0.6929
Epoch 4/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5070 - loss: 0.6931 - val_accuracy: 0.5099 - val_loss: 0.6929
Epoch 5/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5083 - loss: 0.6930 - val_accuracy: 0.5088 - val_loss: 0.6929
Epoch 6/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - accuracy: 0.5078 - loss: 0.6930 - val_accuracy: 0.5068 - val_loss: 0.6929
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
Accuracy Fold 4: 51.08%

--- Fold 5 ---


/home/carl/.local/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15


2026-02-27 14:41:34.160845: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 67464800 exceeds 10% of free system memory.


412/412 ━━━━━━━━━━━━━━━━━━━━ 13s 22ms/step - accuracy: 0.5031 - loss: 0.6938 - val_accuracy: 0.5131 - val_loss: 0.6929
Epoch 2/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - accuracy: 0.5049 - loss: 0.6931 - val_accuracy: 0.5133 - val_loss: 0.6928
Epoch 3/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5053 - loss: 0.6931 - val_accuracy: 0.5133 - val_loss: 0.6929
Epoch 4/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.5059 - loss: 0.6931 - val_accuracy: 0.5133 - val_loss: 0.6929
Epoch 5/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 9s 21ms/step - accuracy: 0.5065 - loss: 0.6930 - val_accuracy: 0.5132 - val_loss: 0.6928
Epoch 6/15
412/412 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - accuracy: 0.5062 - loss: 0.6930 - val_accuracy: 0.5132 - val_loss: 0.6928
52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
Accuracy Fold 5: 51.33%

Accuracy Moyenne de la Validation Croisée : 50.75%


### Inférence et Soumission

In [9]:
print("Génération des prédictions sur l'ensemble de test...")

test_preds = np.zeros(len(X_test_seq))

# On moyenne les prédictions des n_splits modèles
for model in models_nn:
    fold_preds = model.predict(X_test_seq, batch_size=2048).ravel()
    test_preds += fold_preds / n_splits

# Transformation des probabilités moyennes en classes (0 ou 1)
final_predictions = (test_preds > 0.5).astype(int)

# Création du fichier de soumission
submission = pd.DataFrame({
    'target': final_predictions
}, index=sample_submission.index)

submission.to_csv('../submissions/preds_cnn_lstm.csv')
print("Fichier de soumission 'preds_cnn_lstm.csv' généré avec succès !")

Génération des prédictions sur l'ensemble de test...
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Fichier de soumission 'preds_cnn_lstm.csv' généré avec succès !
